# Notebook 089: Dynamic Column Masking for PII Protection and Compliance

**Module:** Module 09: Databricks Platform, Unity Catalog, Security & FinOps  
**Difficulty:** `Intermediate`  
**Databricks Masterclass Reference:** Video 14: Master Unity Catalog & Video 22: PII Data Masterclass  

---

## 1. Business Scenario & Real-World Context
A medical insurance database stores member Social Security Numbers (SSN), full names, and email addresses. Customer service representatives must only see masked SSNs (`***-**-1234`), while compliance officers see cleartext.

## 2. Technical Requirement & Problem Statement
Implement dynamic Column Masking in Databricks Unity Catalog: create a SQL masking function using `is_account_group_member()`, apply the mask to sensitive columns using `ALTER TABLE ... ALTER COLUMN ... SET MASK`, and verify dynamic redaction.

## 3. Spark & Databricks Architecture Notes
Column Masks evaluate dynamically at query execution. If a user is not in the authorized administrative group, the masking function replaces sensitive column data with a masked pattern (e.g. SHA256 hash or asterisks) before returning results.


## 4. Environment Setup & SparkSession Initialization
We configure a resilient `SparkSession` compatible with local PySpark development, Databricks runtime, or CI/CD test runners.

In [ ]:
import os
import sys
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T
from pyspark.sql.window import Window

# Initialize SparkSession with Delta Lake support and optimized local configs
spark = (
    SparkSession.builder
    .appName(f"UseCase_089_Dynamic_Column_Masking_for_PII_Protection_and_Compliance")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.default.parallelism", "4")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print(f" Spark Version: {spark.version}")
print(f" Session Active: {spark.sparkContext.appName}")


## 5. Defining SQL Column Masking Function
Create SQL function that returns cleartext for compliance officers and redacted strings for others.

In [ ]:
# SQL Mask Function Blueprint:
mask_sql = """
CREATE OR REPLACE FUNCTION enterprise_prod.governance.ssn_mask(ssn STRING)
RETURN IF(
    is_account_group_member('compliance_officers'),
    ssn,  -- Cleartext for compliance officers
    CONCAT('***-**-', SUBSTRING(ssn, 8, 4)) -- Redacted for general analysts
);
"""

print("Column Masking SQL Function:")
print(mask_sql)

## 6. Attaching Column Mask to Table Column
Attach mask function using SET MASK DDL.

In [ ]:
# Apply mask to table:
# ALTER TABLE enterprise_prod.hr.t_members
# ALTER COLUMN ssn SET MASK enterprise_prod.governance.ssn_mask;

print("SSN column masked with dynamic security policy.")

## 7. Simulating Dynamic Column Masking in PySpark
Demonstrate masking transformations in standard PySpark.

In [ ]:
members = [
    ("M-101", "Alice Smith", "123-45-6789", "alice@corp.com"),
    ("M-102", "Bob Jones", "987-65-4321", "bob@corp.com")
]
df_members = spark.createDataFrame(members, ["member_id", "name", "ssn", "email"])

# Simulate non-compliance user session
is_compliance = False

df_masked = df_members.withColumn(
    "ssn",
    F.when(F.lit(is_compliance), F.col("ssn"))
     .otherwise(F.concat(F.lit("***-**-"), F.substring(F.col("ssn"), 8, 4)))
).withColumn(
    "email",
    F.when(F.lit(is_compliance), F.col("email"))
     .otherwise(F.regexp_replace(F.col("email"), r"(^.).*(@.*$)", "$1***$2"))
)

print(f"Masked Output for Standard Analyst (is_compliance={is_compliance}):")
df_masked.show(truncate=False)

## 9. Key Architectural Takeaways
- **Column Masking anonymizes sensitive attributes dynamically without mutating underlying storage files.**
- **Single table serves both authorized compliance officers (cleartext) and general analysts (masked).**
- **Column masks cannot be bypassed by copying or selecting into another table without proper privileges.**

## 10. Senior Data Engineering Interview Questions
1. What is the difference between dynamic Column Masking and permanent hashing at write time?
2. Can a user bypass a Column Mask by executing `SELECT * INTO new_table FROM masked_table`?
3. How do Column Masks interact with downstream aggregations (e.g. `COUNT(DISTINCT ssn)`)?

---
*Created for the **Spark Coding Use Cases Masterclass** repository referencing the Databricks All-in-One Playlist.*
